# LAB 1 for CS4243: Textures and Materials
## Notebook 4 — Personal photos and controlled VLM defect analysis

> **Introduction.** This final notebook combines two open-ended studies. Part I
> develops Texture Passports for your own photographs. Part II evaluates a VLM
> such as ChatGPT on five held-out MVTec defects under three controlled levels
> of information. Preserve inputs, prompts, outputs, and limitations.


# Part I — Personal-photo investigation

## Before looking at predictions

Complete `personal_annotations.csv` first. Obtain two independent selections
of three to five DTD terms per photograph, remove EXIF metadata, and check that
no people, identifying documents, or precise locations are visible.


## Suggested investigation structure

1. Introduce the collection with all relevant information.
2. Apply the frozen material, defect, and DTD systems without adapting them to this new data.
3. Select revealing successes, failures, and intermediate response maps.
4. Test one focused question about illumination, scale, or viewpoint.
5. Produce one Texture Passport per image and explain/reason about the uncertainty.


In [ ]:
import sys
from pathlib import Path

import joblib

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PERSONAL_MANIFEST = ROOT / "manifests" / "personal_annotations.csv"
MODEL_DIR = ROOT / "outputs" / "models"

print("manifest available:", PERSONAL_MANIFEST.exists())
print("model directory available:", MODEL_DIR.exists())

# Notebooks 2 and 3 save self-describing bundles here. Load whichever are
# available, while keeping this open-ended notebook usable before training.
task_ab_path = MODEL_DIR / "task_ab_models.joblib"
task_c_path = MODEL_DIR / "task_c_attribute_model.joblib"
task_ab_bundle = joblib.load(task_ab_path) if task_ab_path.exists() else None
task_c_bundle = joblib.load(task_c_path) if task_c_path.exists() else None

if task_ab_bundle is None or task_c_bundle is None:
    print("Run Notebooks 2 and 3 first to create any missing model bundles.")
else:
    material_model = task_ab_bundle["material_classifier"]
    normal_models = task_ab_bundle["normal_models"]
    attribute_model = task_c_bundle["attribute_classifier"]
    feature_config = task_ab_bundle["feature_config"]
    normality_config = task_ab_bundle["normality_config"]
    attribute_config = task_c_bundle["feature_config"]
    attributes = task_c_bundle["attributes"]
    print("Loaded Task A/B keys:", sorted(task_ab_bundle))
    print("Loaded Task C keys:", sorted(task_c_bundle))

## Personal-photo analysis workspace

When Notebooks 2 and 3 have been run, the setup cell exposes `material_model`, `normal_models`, `attribute_model`, their exact feature configurations, the attribute order, and the complete model bundles. Use these frozen objects without refitting on personal images.

Add your own loading, inference, and plotting cells here. Useful visualisations
include a contact sheet grouped by acquisition condition, image/heatmap/mask
triptychs, material confidence versus anomaly score, and human--model DTD-term
overlap. Every figure should support a stated claim.


In [ ]:
# Add your personal-photo exploration here.
# Start small: load one manifest row, run the frozen models, and build one clear figure.

# Part II — Task D: VLM defect classification

This task tests whether a VLM can identify and name surface defects under three
levels of supplied information. The five queries come from held-out MVTec test.
Do not expose the answer key while collecting predictions.


## Experimental conditions

| Level | Information supplied | Question being tested |
|---|---|---|
| **D1 — Generic zero-shot** | Test image and a generic instruction such as “Find any defect in this surface.” | Can the model discover that something is abnormal without domain guidance? |
| **D2 — Named-defect zero-shot** | Test image, material name, and allowed defect names with short verbal definitions; no defect images | Does domain vocabulary help the model recognize and name defects? |
| **D3 — Example-conditioned** | D2 information plus the fixed labelled support set from public validation | Can visual examples transfer a defect concept to new images? |

Use the same VLM, model version, decoding settings, query order, and output
format for all three conditions. Start a fresh conversation for each query and
condition so earlier answers do not leak information.


## 1. Load the five held-out queries

The public query manifest contains paths, material names, and allowed labels.
True labels are stored separately in the staff solution folder.


In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from texturelab.supplied import load_image

MANIFEST_DIR = ROOT / "manifests"

with (MANIFEST_DIR / "task_d_queries.csv").open(newline="", encoding="utf8") as handle:
    queries = list(csv.DictReader(handle))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, query in zip(axes, queries):
    path = (MANIFEST_DIR / query["path"]).resolve()
    ax.imshow(load_image(str(path), (220, 220)))
    ax.set_title(f"{query['id']}: {query['material']}")
    ax.axis("off")
fig.suptitle("Task D held-out query images")
fig.tight_layout()
plt.show()

## 2. D1 — generic zero-shot

Attach one query image and use only a generic prompt. Ask for a binary defect
decision, a short defect name if abnormal, confidence from 0 to 100, and one
sentence of visible evidence. Do not mention the material or allowed labels.

Suggested prompt:

> Inspect this surface image. Is anything visibly abnormal? Return JSON with
> `is_defective`, `defect_name`, `confidence`, and `evidence`.


## 3. D2 — named-defect zero-shot

Provide the material and its allowed defect names with the supplied short
definitions, but no labelled example images. Require exactly one allowed label
or `good`.


In [ ]:
import json

definitions = json.loads(
    (MANIFEST_DIR / "task_d_definitions.json").read_text(encoding="utf8")
)
for query in queries:
    print()
    print(f"{query['id']} — {query['material']}")
    for name, definition in definitions[query["material"]].items():
        print(f"  {name}: {definition}")

## 4. D3 — example-conditioned

Provide the same D2 text plus the fixed support images associated with the
query ID in `task_d_support.csv`. Each support image is labelled and comes only
from public validation. Do not add hand-selected examples or test images.

Ask the VLM to compare the query with all examples and return the same JSON
schema used in D1 and D2.


In [ ]:
with (MANIFEST_DIR / "task_d_support.csv").open(newline="", encoding="utf8") as handle:
    support = list(csv.DictReader(handle))

# Visualise the fixed support set for the first query.
query_id = queries[0]["id"]
examples = [row for row in support if row["query_id"] == query_id]
fig, axes = plt.subplots(1, len(examples), figsize=(16, 3.5))
for ax, example in zip(axes, examples):
    path = (MANIFEST_DIR / example["path"]).resolve()
    ax.imshow(load_image(str(path), (200, 200)))
    ax.set_title(example["defect_name"], fontsize=9)
    ax.axis("off")
fig.suptitle(f"Fixed validation support set for {query_id}")
fig.tight_layout()
plt.show()

## 5. Record predictions before scoring

Create one row per query and condition with: query ID, condition, predicted
label, defective/not-defective decision, confidence, visible evidence, exact
prompt, model/version, and run date. Preserve the raw conversations or exported
screenshots as evidence. Only after all 15 responses are frozen should the
answer key be used to calculate accuracy.


## Analysis ideas

- Compare D1 anomaly discovery with D2/D3 exact-label accuracy.
- Count answers that violate the allowed vocabulary.
- Plot confidence for correct and incorrect predictions by condition.
- Identify queries where definitions help but examples do not, and vice versa.
- Discuss prompt sensitivity, model updates, nondeterminism, and possible prior
  exposure to the public MVTec dataset.
- Do not claim that five images establish general VLM superiority; this is a
  controlled qualitative probe.
